In [1]:
import great_expectations as gx
import pandas as pd

# Veriyi yükle
df = pd.read_csv("temiz_veri/birlesik_veri.csv", index_col='datetime', parse_dates=True)
print("Veri boyutu:", df.shape)
print("Kurulum başarılı, GX versiyonu:", gx.__version__)

Veri boyutu: (34168, 10)
Kurulum başarılı, GX versiyonu: 1.18.2


In [2]:
# GX bağlamı oluştur (dosya sistemi tabanlı)
context = gx.get_context(mode="ephemeral")

# DataFrame'i GX'e tanıt
veri_kaynagi = context.data_sources.add_pandas("pandas_kaynagi")
veri_varlik = veri_kaynagi.add_dataframe_asset("birlesik_veri")
batch_tanim = veri_varlik.add_batch_definition_whole_dataframe("tam_veri")
batch = batch_tanim.get_batch(batch_parameters={"dataframe": df})

print("Veri kaynağı oluşturuldu ✓")

Veri kaynağı oluşturuldu ✓


In [3]:
# Beklenti paketi oluştur
suite = context.suites.add(gx.ExpectationSuite(name="enerji_veri_kalitesi"))

# Kural 1: datetime index boş olmamalı
suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="Global_active_power"))

# Kural 2: Global_active_power negatif olmamalı
suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(
    column="Global_active_power", min_value=0, max_value=20))

# Kural 3: Sıcaklık makul aralıkta olmalı (-30 ile 50 derece)
suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(
    column="temperature_2m", min_value=-30, max_value=50))

# Kural 4: Nem 0-100 arasında olmalı
suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(
    column="relative_humidity_2m", min_value=0, max_value=100))

# Kural 5: Belirli sütunlar mevcut olmalı
suite.add_expectation(gx.expectations.ExpectColumnToExist(column="Global_active_power"))
suite.add_expectation(gx.expectations.ExpectColumnToExist(column="temperature_2m"))
suite.add_expectation(gx.expectations.ExpectColumnToExist(column="precipitation"))

print(f"{len(suite.expectations)} kural tanımlandı ✓")

7 kural tanımlandı ✓


In [4]:
# Doğrulama tanımı oluştur
dogrulama = context.validation_definitions.add(
    gx.ValidationDefinition(
        name="enerji_dogrulama",
        data=batch_tanim,
        suite=suite
    )
)

# Testleri çalıştır
sonuc = dogrulama.run(batch_parameters={"dataframe": df})

# Sonuçları göster
print("=" * 40)
print("VERİ KALİTE TEST SONUÇLARI")
print("=" * 40)
for beklenti_sonuc in sonuc.results:
    durum = "✅ GEÇTI" if beklenti_sonuc.success else "❌ BAŞARISIZ"
    kural = beklenti_sonuc.expectation_config.type
    print(f"{durum} — {kural}")

print("=" * 40)
print("Genel Sonuç:", "✅ BAŞARILI" if sonuc.success else "❌ BAŞARISIZ")

Calculating Metrics:   0%|          | 0/27 [00:00<?, ?it/s]

VERİ KALİTE TEST SONUÇLARI
✅ GEÇTI — expect_column_values_to_not_be_null
✅ GEÇTI — expect_column_values_to_be_between
✅ GEÇTI — expect_column_to_exist
✅ GEÇTI — expect_column_values_to_be_between
✅ GEÇTI — expect_column_to_exist
✅ GEÇTI — expect_column_values_to_be_between
✅ GEÇTI — expect_column_to_exist
Genel Sonuç: ✅ BAŞARILI
